# SFMPE benchmark on Kaggle GPU

1. In Kaggle notebook Settings, select a GPU accelerator and enable Internet for the first dependency installation.
2. Upload `sfmpe_kaggle_bundle.zip` (created by `python scripts/package_kaggle_bundle.py`) as a Kaggle Dataset and attach it to this notebook.
3. Run all cells. The default smoke config uses 40 inference simulations per target (never more than 100) across Gaussian, SIR, StochVol, and all three inference methods.

For the full experiment, change `CONFIG_PATH` below to `benchmark.yaml`. Results are written to `/kaggle/working/sfmpe-benchmark`; download that directory from the notebook output after the run.

In [ ]:
from pathlib import Path
import os
import sys
import zipfile

INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working/sfmpe-project')
archives = sorted(INPUT_ROOT.rglob('sfmpe_kaggle_bundle.zip'))
if not archives:
    raise FileNotFoundError(
        'Attach the sfmpe_kaggle_bundle.zip dataset to this notebook first.'
    )
if len(archives) > 1:
    raise RuntimeError(f'Found multiple SFMPE bundles: {archives}')
WORK_ROOT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(archives[0]) as bundle:
    bundle.extractall(WORK_ROOT)

PROJECT_ROOT = WORK_ROOT
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
os.chdir(PROJECT_ROOT)
print('Project:', PROJECT_ROOT)
print('Configs:', sorted(path.name for path in (PROJECT_ROOT / 'configs').glob('*.yaml')))

In [ ]:
import subprocess
from pathlib import Path

import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        'CUDA is unavailable. In Kaggle notebook Settings, select a GPU accelerator.'
    )
print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__, '| CUDA:', torch.version.cuda)

# Install project runtime dependencies without asking pip to replace Kaggle's CUDA torch.
torch_constraint = Path('/kaggle/working/sfmpe-torch-constraint.txt')
torch_constraint.write_text(f'torch=={torch.__version__}\n', encoding='utf-8')
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '--quiet', '-c', str(torch_constraint),
    '-r', str(PROJECT_ROOT / 'requirements-kaggle.txt'),
])
check = subprocess.check_output([
    sys.executable, '-c',
    'import torch; print(torch.__version__); print(torch.cuda.is_available())',
], text=True).strip().splitlines()
if len(check) != 2 or check[0] != torch.__version__ or check[1].lower() != 'true':
    raise RuntimeError(f'CUDA PyTorch validation failed after dependency setup: {check}')
print('Runtime dependencies installed; the Kaggle CUDA PyTorch version is unchanged.')

In [ ]:
from sfmpe.benchmark.report import export_benchmark_results
from sfmpe.benchmark.runner import run_benchmark

CONFIG_PATH = PROJECT_ROOT / 'configs' / 'benchmark_kaggle_smoke.yaml'
# To run the full configured experiment instead, use:
# CONFIG_PATH = PROJECT_ROOT / 'configs' / 'benchmark.yaml'
OUTPUT_DIR = Path('/kaggle/working/sfmpe-benchmark')

summaries = run_benchmark(CONFIG_PATH, output_dir=OUTPUT_DIR)
report_dir = export_benchmark_results(OUTPUT_DIR)
print(f'Completed {len(summaries)} task/method/budget groups.')
print('Reports:', report_dir)

In [ ]:
from IPython.display import display, Markdown
import shutil

display(Markdown((OUTPUT_DIR / 'benchmark_report.md').read_text(encoding='utf-8')))
print('Files:')
for path in sorted(OUTPUT_DIR.rglob('*')):
    if path.is_file():
        print(path.relative_to(OUTPUT_DIR))

archive_path = Path('/kaggle/working/sfmpe-benchmark-report.zip')
shutil.make_archive(str(archive_path.with_suffix('')), 'zip', OUTPUT_DIR)
print('Downloadable report archive:', archive_path)